In [ ]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

print(f"La root del progetto è: {PROJECT_ROOT}")

In [ ]:
from paths import INV_PATH, TP1_PATH, MODEL_PATH, ROCK_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP1_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + ROCK_PATH

model_name = "Rock1.blend"

Import delle librerie

In [ ]:
import torch
import pandas as pd

from modelaquisition.bl2pina import Blend2Pina
from modelaquisition.bl2msh import Blend2Mesh
from modelaquisition.msh2xdmf import Msh2Xdmf

Fissiamo precisione doppia

In [ ]:
torch.set_default_dtype(torch.float64)

# Problema parametrico in considerazione

Consideriamo il problema parametrico sul dominio $\Omega$ di frontiera $\Gamma = \partial \Omega$
$$
\begin{equation}
    \begin{cases}
        \frac{\partial^2 u}{\partial x^2} \left(x, y, z\right) + \frac{\partial^2 u}{\partial y^2} \left(x, y, z \right) + \frac{\partial^2 u}{\partial z^2} \left(x, y, z\right) = - \left( \alpha^2 + \beta^2 \right) \pi^2 \lambda x \cos\left( \alpha \pi y\right) \sin\left( \beta \pi z\right) & \left(x, y, z \right) \in \Omega \\
        u \left( x, y, z \right) = \lambda x \cos\left( \alpha \pi y \right) \sin \left( \beta \pi z \right) & \left( x, y, z \right) \in \Gamma
    \end{cases}
    \tag{1}
\end{equation}
$$

di soluzione analitica
$$
\begin{equation}
    u \left( x, y, z \right) = \lambda x \cos\left( \alpha \pi y \right) \sin \left( \beta \pi z \right) \qquad \left( x, y, z \right) \in \bar{\Omega}
    \tag{2}
\end{equation}
$$

## Creazione dei dati per il problema inverso

Fissiamo i punti in cui sono installati i sensori

In [ ]:
rock = Blend2Pina(LOAD_MODEL + model_name)

Acquisizione dei punti al contrno

In [ ]:
num_points = 500

surface = rock.boundary()
points = surface.sample(num_points)

Fissimao i parametri e collezioniamo i dati simulati

In [ ]:
par_lambda = .1
par_alpha = .2
par_beta = .5

u = par_lambda * points.extract("x").tensor * torch.cos(par_alpha * torch.pi * points.extract("y").tensor) * torch.sin(par_beta * torch.pi * points.extract("z").tensor)

Creazione file .csv per conservare i punti al contorno

In [ ]:
total_info = torch.concat(
    [points.tensor, u],
    1
)

df = pd.DataFrame(
    data=total_info.numpy(),
    columns=["x", "y", "z", "u"]
)

df.to_csv("./files/data.csv", sep=";")

## Creazione mesh e xdmf

In [ ]:
rock_msh = Blend2Mesh(LOAD_MODEL + model_name, "rock")

In [ ]:
rock_msh.create_mesh(len_msh=0.07)

In [ ]:
rock_xdmf = Msh2Xdmf("rock.msh", "rock")
rock_xdmf.to_xdmf()